In [1]:
import numpy as np

class Yahtzee():

    def __init__(self, pmfs, count_mapper, num_games = 10, SEED=0):
        self.count = np.zeros(6)
        self.rng = np.random.default_rng(SEED)
        self.pmfs = pmfs
        self.selected_pmfs = {}
        self.curr_tuple = (0,0)
        self.categories = list(self.pmfs.keys())
        self.count_mapper = count_mapper
        self.upper = [0,0,0,0,0,0]
        self.lower = [0,0,0,0,0,0,0]
        self.score = 0
        self.choices = {
            'ones': True,
            'twos': True,
            'threes': True,
            'fours': True,
            'fives': True,
            'sixes': True,
            'three_of_a_kind': True,
            'four_of_a_kind': True,
            'full_house': True,
            'small_straight': True,
            'large_straight': True,
            'yahtzee': True,
            'chance': True,
            }
        self.game_scores = []
        self.num_games = num_games
        self.num_turns = 0
        self.bonus_recieved = 0
        self.upper_scores = []

    def run_yahtzee(self, games):
        self.num_games = games
        for i in range(self.num_games):
            # print(f'GAME #{i + 1}')
            if i % 100 == 0:
                print(i)
            self.clean()
            while self.num_turns < 13:
                self.turn()
            self.selected_pmfs = {}
            if sum(self.upper) >= 63:
                self.bonus_recieved += 1
                self.score += 35
            self.game_scores.append(self.score)
            self.upper_scores.append(sum(self.upper))

        return self.game_scores, self.bonus_recieved, self.upper_scores
        
    def turn(self):
        # first roll
        self.count = self.random_generate()
        self.get_pmfs(2)
        self.pick_category(2)
        # print(f"ROLL 1: {self.count}, Best Category: {self.best_category}, Dice Kept: {self.pmfs[self.best_category][1][tuple((self.count_mapper[self.count], 2))]}")
        
        num_new_dice = self.update_count(2)
        
        if num_new_dice == 0:
            self.end_turn()
        else:
            # second roll
            self.count = self.combine_new_roll(num_new_dice)
            self.get_pmfs(1)
            self.pick_category(1)
            # print(f"ROLL 2: {self.count}, Best Category: {self.best_category}, Dice Kept: {self.pmfs[self.best_category][1][tuple((self.count_mapper[self.count], 1))]}")
            num_new_dice = self.update_count(1)

            if num_new_dice == 0:
                self.end_turn()
            else:
                # third roll
                self.count = self.combine_new_roll(num_new_dice)
                self.get_pmfs(0)
                self.pick_category(0)
                # print(f"ROLL 3: {self.count}")
                self.end_turn()                


    def end_turn(self):
        new_score = list(self.selected_pmfs[self.best_category].keys())[0]
        self.choices[self.best_category] = False
        if self.best_category == 'ones':
            self.upper[0] = new_score
            self.score += new_score
        elif self.best_category == 'twos':
            new_score = new_score * 2
            self.upper[1] = new_score
            self.score += new_score
        elif self.best_category == 'threes':
            new_score = new_score * 3
            self.upper[2] = new_score
            self.score += new_score
        elif self.best_category == 'fours':
            new_score = new_score * 4
            self.upper[3] = new_score
            self.score += new_score
        elif self.best_category == 'fives':
            new_score = new_score * 5
            self.upper[4] = new_score
            self.score += new_score
        elif self.best_category == 'sixes':
            new_score = new_score * 6
            self.upper[5] = new_score
            self.score += new_score
        else:
            self.score += new_score
        self.num_turns += 1
        # print(f"Best Category = {self.best_category}, dice = {self.count}, Turn Score = {new_score}, num turns = {self.num_turns}")
            
    def get_pmfs(self, roll_num):
        self.selected_pmfs = {}
        the_tuple = tuple((self.count_mapper[self.count], roll_num))
        self.curr_tuple = the_tuple
        
        for category in self.categories:
            pmf = self.pmfs[category][0][self.curr_tuple]
            self.selected_pmfs[category] = pmf

    def pick_category(self, num_rolls):
        max_category = ''
        max_expected = -1
        for category in self.categories:
            F_expected = self.calculate_expected(category)
            if F_expected > max_expected and self.choices[category] == True:              
                max_expected = F_expected
                max_category = category
        
        self.best_category = max_category

    def update_count(self, roll_num):
        dice_kept = self.pmfs[self.best_category][1][self.curr_tuple]
        self.dice_kept = dice_kept
        num_new_dice = 5 - sum(dice_kept)
        return num_new_dice

    def combine_new_roll(self, num_new_rolls):
        new_count = self.random_generate(num_new_rolls)
        numbers = tuple(a + b for a, b in zip(self.dice_kept, new_count))
        return numbers

    def check_chance_expected(self, num_rolls):
        if num_rolls == 0:
            if list(self.selected_pmfs['chance'].keys())[0] < 23.36:
                return False
            else: 
                return True
        else:
            chance_pmf = self.selected_pmfs['chance']
            curr_keys = chance_pmf.keys()
            expected = round(sum(chance_pmf[key] * key for key in curr_keys), 2)
            if expected < 23.36:
                return False
            else:
                return True

    def calculate_expected(self, category):
        keys = self.selected_pmfs[category].keys()
        multiplier = 1
        if category == 'twos':
            multiplier = 2
        if category == 'threes':
            multiplier = 3
        if category == 'fours':
            multiplier = 4
        if category == 'fives':
            multiplier = 5
        if category == 'sixes':
            multiplier = 6

        F_expected = sum([self.selected_pmfs[category][key] * key * multiplier for key in keys])
        return F_expected
        
    def clean(self):
        for category in self.categories:
            self.choices[category] = True
        self.num_turns = 0
        self.score = 0
        self.upper = np.zeros(6)
        
    def random_generate(self, num=5):
        numbers = self.rng.integers(0,6,num)
        count = np.zeros(6)
        for number in numbers:
            count[number] += 1
        return tuple(count)
        

In [2]:
# LOAD DATASETS

import os
import pickle

folder = "all_pmfs"  
# folder = "all_greedy"
loaded = {}

for fname in os.listdir(folder):
    if fname.endswith(".pkl"):
        varname = fname.replace("_DP_2.pkl", "")
        # varname = fname.replace("_greedy2.pkl", "")
        with open(os.path.join(folder, fname), "rb") as f:
            loaded[varname] = pickle.load(f)

with open('count_mapper.pkl', 'rb') as f:
    counter_mapper = pickle.load(f)

In [3]:
DP_Games = Yahtzee(loaded, counter_mapper)
gamescores, bonuses, upper_scores = DP_Games.run_yahtzee(10000)
results = np.array([gamescores, bonuses, upper_scores], dtype=object)
# print(f"Final score: {results[0][0]}")
# np.save('greedy_results.npy', results)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
